<a href="https://colab.research.google.com/github/Subuktageen-Farooqi/ms_course_deeplearning/blob/main/ms_deeplearning_tutorial_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 14_A — Simple RNN for Named Entity Recognition (NER)

## TensorFlow Implementation

In [2]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout, TimeDistributed
from sklearn.preprocessing import LabelEncoder

# Sample data
sentences = [
    "Barack Obama was born in Hawaii",
    "Google is based in Mountain View"
]

labels = [
    ["PERSON", "PERSON", "O", "O", "O", "LOCATION", "O"],  # Labels for the first sentence
    ["ORGANIZATION", "O", "O", "O", "LOCATION", "O"]  # Labels for the second sentence
]

# Tokenizing the sentences (converting words into integers)
tokenizer = Tokenizer(lower=True)
tokenizer.fit_on_texts(sentences)
X = tokenizer.texts_to_sequences(sentences)

# Padding the sequences to have the same length
X = pad_sequences(X, padding='post')

# Encode the labels
label_encoder = LabelEncoder()
label_encoder.fit(["O", "PERSON", "LOCATION", "ORGANIZATION"])

# Convert labels to numerical values (e.g., O = 0, PERSON = 1)
y = [label_encoder.transform(label) for label in labels]

# Pad the labels so that they match the shape of the input sequences (X)
y = pad_sequences(y, padding='post', maxlen=X.shape[1])

# Reshape y to match the shape of the input sequence (for time-step labeling)
y = np.expand_dims(y, -1)  # Expanding dimensions to match the model's output

# Model definition
model = Sequential()

# Embedding layer: Convert words into dense vectors
model.add(Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=50, input_length=X.shape[1]))

# Simple RNN layer
model.add(SimpleRNN(units=50, return_sequences=True))

# Dropout to avoid overfitting
model.add(Dropout(0.1))

# TimeDistributed Dense layer for making predictions at each time step
model.add(TimeDistributed(Dense(len(label_encoder.classes_), activation='softmax')))

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(np.array(X), np.array(y), epochs=10, batch_size=2)

# Test with a new sentence
test_sentence = ["Barack Obama went to Hawaii"]
test_sequence = tokenizer.texts_to_sequences(test_sentence)
test_sequence = pad_sequences(test_sequence, padding='post', maxlen=X.shape[1])

# Predicting the NER labels for the test sentence
predictions = model.predict(test_sequence)

# Decode predictions
decoded_predictions = label_encoder.inverse_transform(np.argmax(predictions, axis=-1)[0])

# Display results
for word, label in zip(test_sentence[0].split(), decoded_predictions):
    print(f"Word: {word} - Predicted Label: {label}")

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.2500 - loss: 1.4256
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.3333 - loss: 1.3915
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.5000 - loss: 1.3574
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.4167 - loss: 1.3422
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.6667 - loss: 1.3009
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.8333 - loss: 1.2787
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.9167 - loss: 1.2413
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.9167 - loss: 1.2272
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.9167 - loss: 1.1943
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.9167 - loss: 1.1720
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step
Word: Barack - Predicted Label: PERSON
Word: Obama - Predicted Label: O
Word: went - Predicted Label: O
Word: to - Predicted Label: O
Word: Haw

## PyTorch Implementation

In [3]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [4]:
# Same small tutorial-style dataset, but with token-level sentence lists for safer alignment.
toy_data = [
    (
        ["Barack", "Obama", "was", "born", "in", "Hawaii"],
        ["PERSON", "PERSON", "O", "O", "O", "LOCATION"]
    ),
    (
        ["Google", "is", "based", "in", "Mountain", "View"],
        ["ORGANIZATION", "O", "O", "O", "LOCATION", "LOCATION"]
    ),
]

label_names = ["PAD", "O", "PERSON", "LOCATION", "ORGANIZATION"]
label_to_id = {label: idx for idx, label in enumerate(label_names)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

def build_vocab(dataset):
    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for words, labels in dataset:
        assert len(words) == len(labels), "Each word must have exactly one label."
        for word in words:
            word_lower = word.lower()
            if word_lower not in vocab:
                vocab[word_lower] = len(vocab)
    return vocab

vocab = build_vocab(toy_data)

def encode_words(words, vocab):
    return [vocab.get(word.lower(), vocab[UNK_TOKEN]) for word in words]

def encode_labels(labels):
    return [label_to_id[label] for label in labels]

class NERDataset(Dataset):
    def __init__(self, examples, vocab):
        self.examples = examples
        self.vocab = vocab

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        words, labels = self.examples[idx]
        input_ids = encode_words(words, self.vocab)
        label_ids = encode_labels(labels)
        assert len(input_ids) == len(label_ids), "Each word must have one label."
        return {
            "words": words,
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(label_ids, dtype=torch.long)
        }

def make_collate_fn(vocab, label_to_id):
    def collate_ner_batch(batch):
        max_len = max(item["input_ids"].shape[0] for item in batch)
        padded_input_ids = []
        padded_labels = []
        lengths = []
        words = []
        for item in batch:
            input_ids = item["input_ids"]
            labels = item["labels"]
            length = input_ids.shape[0]
            pad_len = max_len - length
            padded_input_ids.append(torch.cat([
                input_ids,
                torch.full((pad_len,), vocab[PAD_TOKEN], dtype=torch.long)
            ]))
            padded_labels.append(torch.cat([
                labels,
                torch.full((pad_len,), label_to_id["PAD"], dtype=torch.long)
            ]))
            lengths.append(length)
            words.append(item["words"])
        return {
            "words": words,
            "input_ids": torch.stack(padded_input_ids),
            "labels": torch.stack(padded_labels),
            "lengths": torch.tensor(lengths, dtype=torch.long)
        }
    return collate_ner_batch

class SimpleRNNNER(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_labels, pad_idx, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_labels)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        rnn_out, _ = self.rnn(x)
        rnn_out = self.dropout(rnn_out)
        logits = self.classifier(rnn_out)
        return logits

def token_accuracy(logits, labels):
    preds = logits.argmax(dim=-1)
    mask = labels != label_to_id["PAD"]
    correct = (preds[mask] == labels[mask]).sum().item()
    total = mask.sum().item()
    return correct / total if total > 0 else 0.0

toy_loader = DataLoader(
    NERDataset(toy_data, vocab),
    batch_size=2,
    shuffle=True,
    collate_fn=make_collate_fn(vocab, label_to_id)
)

toy_model = SimpleRNNNER(
    vocab_size=len(vocab),
    embedding_dim=50,
    hidden_dim=50,
    num_labels=len(label_names),
    pad_idx=vocab[PAD_TOKEN],
    dropout=0.1
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=label_to_id["PAD"])
optimizer = torch.optim.Adam(toy_model.parameters(), lr=1e-3)

epochs = 20
for epoch in range(epochs):
    toy_model.train()
    total_loss = 0.0
    total_acc = 0.0
    for batch in toy_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        optimizer.zero_grad()
        logits = toy_model(input_ids)
        assert logits.shape[:2] == labels.shape, "Model output must align with token labels."
        loss = criterion(logits.view(-1, logits.shape[-1]), labels.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_acc += token_accuracy(logits.detach(), labels)
    print(f"Epoch [{epoch+1}/{epochs}] Loss: {total_loss/len(toy_loader):.4f} | Token Accuracy: {total_acc/len(toy_loader):.4f}")

Epoch [1/20] Loss: 1.6960 | Token Accuracy: 0.2500
Epoch [2/20] Loss: 1.6350 | Token Accuracy: 0.1667
Epoch [3/20] Loss: 1.5728 | Token Accuracy: 0.4167
Epoch [4/20] Loss: 1.4815 | Token Accuracy: 0.5000
Epoch [5/20] Loss: 1.4569 | Token Accuracy: 0.4167
Epoch [6/20] Loss: 1.4233 | Token Accuracy: 0.5833
Epoch [7/20] Loss: 1.3205 | Token Accuracy: 0.6667
Epoch [8/20] Loss: 1.3411 | Token Accuracy: 0.6667
Epoch [9/20] Loss: 1.2534 | Token Accuracy: 0.7500
Epoch [10/20] Loss: 1.1864 | Token Accuracy: 0.7500
Epoch [11/20] Loss: 1.1613 | Token Accuracy: 0.8333
Epoch [12/20] Loss: 1.0971 | Token Accuracy: 0.6667
Epoch [13/20] Loss: 1.0334 | Token Accuracy: 0.8333
Epoch [14/20] Loss: 0.9832 | Token Accuracy: 0.9167
Epoch [15/20] Loss: 0.9890 | Token Accuracy: 0.9167
Epoch [16/20] Loss: 0.9117 | Token Accuracy: 1.0000
Epoch [17/20] Loss: 0.8669 | Token Accuracy: 1.0000
Epoch [18/20] Loss: 0.7898 | Token Accuracy: 1.0000
Epoch [19/20] Loss: 0.8310 | Token Accuracy: 1.0000
Epoch [20/20] Loss: 0

In [5]:
def predict_ner(model, sentence_words, vocab):
    model.eval()
    input_ids = torch.tensor([encode_words(sentence_words, vocab)], dtype=torch.long).to(device)
    with torch.no_grad():
        logits = model(input_ids)
        pred_ids = logits.argmax(dim=-1).squeeze(0).cpu().tolist()
    return [(word, id_to_label[pred_id]) for word, pred_id in zip(sentence_words, pred_ids)]

predict_ner(toy_model, ["Barack", "Obama", "went", "to", "Hawaii"], vocab)

[('Barack', 'PERSON'),
 ('Obama', 'PERSON'),
 ('went', 'ORGANIZATION'),
 ('to', 'ORGANIZATION'),
 ('Hawaii', 'LOCATION')]

## Task 01 - Make your own Dataset and Test it with the Model

In [6]:
custom_data = [
    (["Ali", "works", "at", "Google", "in", "Lahore"], ["PERSON", "O", "O", "ORGANIZATION", "O", "LOCATION"]),
    (["Sara", "joined", "Microsoft", "in", "Karachi"], ["PERSON", "O", "ORGANIZATION", "O", "LOCATION"]),
    (["Ahmed", "visited", "Islamabad", "yesterday"], ["PERSON", "O", "LOCATION", "O"]),
    (["OpenAI", "is", "based", "in", "San", "Francisco"], ["ORGANIZATION", "O", "O", "O", "LOCATION", "LOCATION"]),
    (["Fatima", "met", "Bilal", "in", "Rawalpindi"], ["PERSON", "O", "PERSON", "O", "LOCATION"]),
    (["Tesla", "opened", "an", "office", "in", "Dubai"], ["ORGANIZATION", "O", "O", "O", "O", "LOCATION"]),
    (["Hassan", "studies", "at", "NUST"], ["PERSON", "O", "O", "ORGANIZATION"]),
    (["Apple", "hired", "Ayesha", "from", "Lahore"], ["ORGANIZATION", "O", "PERSON", "O", "LOCATION"]),
    (["Bilal", "moved", "to", "Peshawar"], ["PERSON", "O", "O", "LOCATION"]),
    (["Meta", "has", "an", "office", "in", "London"], ["ORGANIZATION", "O", "O", "O", "O", "LOCATION"]),
    (["Zara", "works", "with", "Amazon"], ["PERSON", "O", "O", "ORGANIZATION"]),
    (["IBM", "invited", "Omar", "to", "Paris"], ["ORGANIZATION", "O", "PERSON", "O", "LOCATION"]),
]

# Simple fixed split.
# Train data is used for learning.
# Validation data is used for monitoring hyperparameters.
# Test data is held out until final evaluation.
train_data = custom_data[:8]
val_data = custom_data[8:10]
test_data = custom_data[10:]

custom_vocab = build_vocab(train_data)
custom_collate = make_collate_fn(custom_vocab, label_to_id)

train_loader = DataLoader(NERDataset(train_data, custom_vocab), batch_size=4, shuffle=True, collate_fn=custom_collate)
val_loader = DataLoader(NERDataset(val_data, custom_vocab), batch_size=2, shuffle=False, collate_fn=custom_collate)
test_loader = DataLoader(NERDataset(test_data, custom_vocab), batch_size=2, shuffle=False, collate_fn=custom_collate)

print("Vocabulary size:", len(custom_vocab))
custom_vocab

Vocabulary size: 37


{'<PAD>': 0,
 '<UNK>': 1,
 'ali': 2,
 'works': 3,
 'at': 4,
 'google': 5,
 'in': 6,
 'lahore': 7,
 'sara': 8,
 'joined': 9,
 'microsoft': 10,
 'karachi': 11,
 'ahmed': 12,
 'visited': 13,
 'islamabad': 14,
 'yesterday': 15,
 'openai': 16,
 'is': 17,
 'based': 18,
 'san': 19,
 'francisco': 20,
 'fatima': 21,
 'met': 22,
 'bilal': 23,
 'rawalpindi': 24,
 'tesla': 25,
 'opened': 26,
 'an': 27,
 'office': 28,
 'dubai': 29,
 'hassan': 30,
 'studies': 31,
 'nust': 32,
 'apple': 33,
 'hired': 34,
 'ayesha': 35,
 'from': 36}

In [7]:
def evaluate_ner(model, data_loader, loss_fn=None):
    if loss_fn is None:
        loss_fn = nn.CrossEntropyLoss(ignore_index=label_to_id["PAD"])
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            logits = model(input_ids)
            loss = loss_fn(logits.view(-1, logits.shape[-1]), labels.view(-1))
            total_loss += loss.item()
            total_acc += token_accuracy(logits, labels)
    return total_loss / len(data_loader), total_acc / len(data_loader)

def train_ner_model(train_loader, val_loader, vocab_size, pad_idx, hidden_dim=50, epochs=20, lr=1e-3):
    model = SimpleRNNNER(
        vocab_size=vocab_size,
        embedding_dim=50,
        hidden_dim=hidden_dim,
        num_labels=len(label_names),
        pad_idx=pad_idx,
        dropout=0.1
    ).to(device)
    local_criterion = nn.CrossEntropyLoss(ignore_index=label_to_id["PAD"])
    local_optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0
        total_train_acc = 0.0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            local_optimizer.zero_grad()
            logits = model(input_ids)
            assert logits.shape[:2] == labels.shape, "Model output must align with token labels."
            loss = local_criterion(logits.view(-1, logits.shape[-1]), labels.view(-1))
            loss.backward()
            local_optimizer.step()
            total_train_loss += loss.item()
            total_train_acc += token_accuracy(logits.detach(), labels)
        train_loss = total_train_loss / len(train_loader)
        train_acc = total_train_acc / len(train_loader)
        val_loss, val_acc = evaluate_ner(model, val_loader, local_criterion)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    return model, history

custom_model, custom_history = train_ner_model(
    train_loader=train_loader,
    val_loader=val_loader,
    vocab_size=len(custom_vocab),
    pad_idx=custom_vocab[PAD_TOKEN],
    hidden_dim=50,
    epochs=30,
    lr=1e-3
)

test_loss, test_acc = evaluate_ner(custom_model, test_loader)
print(f"Final Test Loss: {test_loss:.4f} | Final Test Token Accuracy: {test_acc:.4f}")

Epoch [1/30] Train Loss: 1.7677 | Train Acc: 0.0964 | Val Loss: 1.8361 | Val Acc: 0.1000
Epoch [2/30] Train Loss: 1.6797 | Train Acc: 0.1464 | Val Loss: 1.8038 | Val Acc: 0.1000
Epoch [3/30] Train Loss: 1.6122 | Train Acc: 0.1699 | Val Loss: 1.7727 | Val Acc: 0.1000
Epoch [4/30] Train Loss: 1.5520 | Train Acc: 0.3433 | Val Loss: 1.7429 | Val Acc: 0.1000
Epoch [5/30] Train Loss: 1.4776 | Train Acc: 0.3917 | Val Loss: 1.7158 | Val Acc: 0.1000
Epoch [6/30] Train Loss: 1.4153 | Train Acc: 0.4881 | Val Loss: 1.6904 | Val Acc: 0.2000
Epoch [7/30] Train Loss: 1.3611 | Train Acc: 0.5881 | Val Loss: 1.6662 | Val Acc: 0.2000
Epoch [8/30] Train Loss: 1.3152 | Train Acc: 0.7833 | Val Loss: 1.6426 | Val Acc: 0.3000
Epoch [9/30] Train Loss: 1.2587 | Train Acc: 0.6722 | Val Loss: 1.6201 | Val Acc: 0.3000
Epoch [10/30] Train Loss: 1.2090 | Train Acc: 0.7560 | Val Loss: 1.5994 | Val Acc: 0.3000
Epoch [11/30] Train Loss: 1.1367 | Train Acc: 0.8024 | Val Loss: 1.5801 | Val Acc: 0.3000
Epoch [12/30] Train

In [8]:
# Test the model on custom unseen-style sentences.
test_sentences = [
    ["Ali", "joined", "Amazon", "in", "Dubai"],
    ["Ayesha", "visited", "Islamabad"],
    ["OpenAI", "hired", "Fatima"]
]

for sent in test_sentences:
    print(predict_ner(custom_model, sent, custom_vocab))
    print("-" * 60)

[('Ali', 'PERSON'), ('joined', 'O'), ('Amazon', 'PERSON'), ('in', 'O'), ('Dubai', 'LOCATION')]
------------------------------------------------------------
[('Ayesha', 'PERSON'), ('visited', 'O'), ('Islamabad', 'LOCATION')]
------------------------------------------------------------
[('OpenAI', 'ORGANIZATION'), ('hired', 'O'), ('Fatima', 'O')]
------------------------------------------------------------


## Task 2 — Change Units, Epochs, and Learning Rate

In [9]:
sweep_configs = [
    {"hidden_dim": 32, "epochs": 20, "lr": 1e-3},
    {"hidden_dim": 50, "epochs": 30, "lr": 1e-3},
    {"hidden_dim": 64, "epochs": 30, "lr": 5e-4},
]

sweep_results = []

for cfg in sweep_configs:
    print("=" * 80)
    print(f"Training config: {cfg}")
    model, history = train_ner_model(
        train_loader=train_loader,
        val_loader=val_loader,
        vocab_size=len(custom_vocab),
        pad_idx=custom_vocab[PAD_TOKEN],
        hidden_dim=cfg["hidden_dim"],
        epochs=cfg["epochs"],
        lr=cfg["lr"]
    )
    sweep_results.append({
        "hidden_dim": cfg["hidden_dim"],
        "epochs": cfg["epochs"],
        "lr": cfg["lr"],
        "final_val_loss": history["val_loss"][-1],
        "final_val_acc": history["val_acc"][-1],
        "model": model
    })

summary_rows = [{k: v for k, v in row.items() if k != "model"} for row in sweep_results]
summary_rows

Training config: {'hidden_dim': 32, 'epochs': 20, 'lr': 0.001}
Epoch [1/20] Train Loss: 1.5640 | Train Acc: 0.3167 | Val Loss: 1.5617 | Val Acc: 0.4000
Epoch [2/20] Train Loss: 1.5551 | Train Acc: 0.3417 | Val Loss: 1.5316 | Val Acc: 0.4000
Epoch [3/20] Train Loss: 1.4858 | Train Acc: 0.4270 | Val Loss: 1.5033 | Val Acc: 0.6000
Epoch [4/20] Train Loss: 1.4534 | Train Acc: 0.4713 | Val Loss: 1.4751 | Val Acc: 0.6000
Epoch [5/20] Train Loss: 1.4031 | Train Acc: 0.5143 | Val Loss: 1.4469 | Val Acc: 0.7000
Epoch [6/20] Train Loss: 1.3688 | Train Acc: 0.6077 | Val Loss: 1.4212 | Val Acc: 0.7000
Epoch [7/20] Train Loss: 1.3294 | Train Acc: 0.6095 | Val Loss: 1.3947 | Val Acc: 0.7000
Epoch [8/20] Train Loss: 1.2782 | Train Acc: 0.7029 | Val Loss: 1.3686 | Val Acc: 0.7000
Epoch [9/20] Train Loss: 1.2402 | Train Acc: 0.7249 | Val Loss: 1.3447 | Val Acc: 0.7000
Epoch [10/20] Train Loss: 1.2018 | Train Acc: 0.7560 | Val Loss: 1.3195 | Val Acc: 0.8000
Epoch [11/20] Train Loss: 1.1453 | Train Acc: 

[{'hidden_dim': 32,
  'epochs': 20,
  'lr': 0.001,
  'final_val_loss': 1.0976333618164062,
  'final_val_acc': 0.8},
 {'hidden_dim': 50,
  'epochs': 30,
  'lr': 0.001,
  'final_val_loss': 0.8210590481758118,
  'final_val_acc': 0.8},
 {'hidden_dim': 64,
  'epochs': 30,
  'lr': 0.0005,
  'final_val_loss': 0.9359265565872192,
  'final_val_acc': 0.7}]

In [10]:
import pandas as pd

sweep_df = pd.DataFrame(summary_rows)
sweep_df

,hidden_dim,epochs,lr,final_val_loss,final_val_acc
0,32,20,0.0010,1.097633,0.8
1,50,30,0.0010,0.821059,0.8
2,64,30,0.0005,0.935927,0.7


In [14]:
# Select best configuration by highest validation accuracy, then lowest validation loss.
best_row = sorted(sweep_results, key=lambda row: (-row["final_val_acc"], row["final_val_loss"]))[0]
best_model = best_row["model"]
final_test_loss, final_test_acc = evaluate_ner(best_model, test_loader)

print("Best validation config:\n")
for k, v in best_row.items():
    if k != "model":
        print(f"{k}: {v}")
print(f"\nHeld-out Test Loss: {final_test_loss:.4f}")
print(f"Held-out Test Token Accuracy: {final_test_acc:.4f}")

Best validation config:

hidden_dim: 50
epochs: 30
lr: 0.001
final_val_loss: 0.8210590481758118
final_val_acc: 0.8

Held-out Test Loss: 1.3122
Held-out Test Token Accuracy: 0.4444
